# Notebook 03 — Classification (Checkpoint 4)

**Binned target:** `price_tier` = {budget, mid, premium} from **tertiles** of `price`.

**Why tertiles?**  
Three classes, roughly balanced, easy to explain. Domain-specific bins would be cool too, but I wanted something reproducible on a homework timeline.


In [1]:
import pandas as pd  #I'm doing this because pandas loads the cleaned dataset with the price tier labels.
import numpy as np  #I'm doing this because numpy is available if we need numeric helpers in later edits.
import matplotlib.pyplot as plt  #I'm doing this because matplotlib hosts the confusion matrix axes.
from sklearn.compose import ColumnTransformer  #I'm doing this because the same numeric and categorical preprocessing applies as in regression.
from sklearn.ensemble import HistGradientBoostingClassifier  #I'm doing this because histogram-based boosting handles mixed data with nonlinear boundaries.
from sklearn.linear_model import LogisticRegression  #I'm doing this because multinomial logistic regression is a strong linear multiclass baseline.
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score  #I'm doing this because these metrics summarize multiclass performance.
from sklearn.model_selection import train_test_split  #I'm doing this because stratified splitting preserves class proportions in train and test.
from sklearn.pipeline import Pipeline  #I'm doing this because pipelines keep preprocessing and classifier fitted together.
from sklearn.preprocessing import OneHotEncoder, StandardScaler  #I'm doing this because scaling helps logistic regression and one-hot encodes categories.
import joblib  #I'm doing this because joblib loads the regression bundle for shared metadata and saves the classifier bundle.
from pathlib import Path  #I'm doing this because pathlib locates files under the NEW project root.

PROJECT_ROOT = Path("/Users/jahbrae/Downloads/NEW")  #I'm doing this because all paths resolve relative to your NEW deliverable folder.
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "cleaned_data.csv")  #I'm doing this because classification uses the same cleaned table as regression.
FEATURES = ["vehicle_age", "odometer", "manufacturer", "fuel", "transmission", "drive", "type"]  #I'm doing this because features match the regression design for comparability.
TARGET = "price_tier"  #I'm doing this because TARGET names the three-class label derived from price tertiles.

X = df[FEATURES]  #I'm doing this because X contains predictors for the classifier.
y = df[TARGET]  #I'm doing this because y holds budget, mid, or premium labels.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)  #I'm doing this because stratify=y keeps similar class counts in train and test.

def make_pre():  #I'm doing this because the helper rebuilds the ColumnTransformer for each pipeline.
    return ColumnTransformer(  #I'm doing this because returning the object allows reuse in both classifier pipelines.
        [  #I'm doing this because ColumnTransformer expects a list of (name, transformer, columns) triples.
            ("num", StandardScaler(), ["vehicle_age", "odometer"]),  #I'm doing this because logistic regression benefits from scaled continuous inputs.
            (  #I'm doing this because this opens the tuple for the categorical preprocessing branch.
                "cat",  #I'm doing this because this labels the categorical sub-transformer inside the ColumnTransformer.
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),  #I'm doing this because dense encoding feeds both logistic and histogram models here.
                ["manufacturer", "fuel", "transmission", "drive", "type"],  #I'm doing this because these fields are categorical factors of price tier.
            ),  #I'm doing this because this closes the categorical branch tuple.
        ]  #I'm doing this because this closes the transformer list for ColumnTransformer.
    )  #I'm doing this because this closes the ColumnTransformer constructor returned by make_pre.

clfs = {  #I'm doing this because a dict stores competing classifiers under short keys.
    "logreg": Pipeline([("pre", make_pre()), ("m", LogisticRegression(max_iter=2000, random_state=42))]),  #I'm doing this because high max_iter avoids convergence warnings on wide one-hot data.
    "hgb": Pipeline([("pre", make_pre()), ("m", HistGradientBoostingClassifier(max_depth=10, random_state=42))]),  #I'm doing this because depth ten allows interaction depth without extreme overfitting.
}  #I'm doing this because closing the dict completes the model list.

for name, pipe in clfs.items():  #I'm doing this because the loop trains and evaluates each classifier with identical splits.
    pipe.fit(X_train, y_train)  #I'm doing this because fit learns preprocessor encodings and classifier weights on training data.
    pred = pipe.predict(X_test)  #I'm doing this because held-out predictions measure generalization error.
    print("===", name, "===")  #I'm doing this because a clear banner separates console output per model.
    print("acc", accuracy_score(y_test, pred))  #I'm doing this because accuracy is an intuitive fraction correct.
    print("f1_weighted", f1_score(y_test, pred, average="weighted"))  #I'm doing this because weighted F1 accounts for class imbalance better than raw accuracy alone.
    print(confusion_matrix(y_test, pred, labels=["budget", "mid", "premium"]))  #I'm doing this because the matrix shows which tiers get confused.
    print(classification_report(y_test, pred))  #I'm doing this because per-class precision and recall explain errors beyond a single score.


=== logreg ===
acc 0.7763303706719136
f1_weighted 0.7766845827171991
[[12171  2311   261]
 [ 2400 10105  2234]
 [  142  2542 12051]]


              precision    recall  f1-score   support

      budget       0.83      0.83      0.83     14743
         mid       0.68      0.69      0.68     14739
     premium       0.83      0.82      0.82     14735

    accuracy                           0.78     44217
   macro avg       0.78      0.78      0.78     44217
weighted avg       0.78      0.78      0.78     44217



=== hgb ===
acc 0.8232580229323563
f1_weighted 0.8235969505615615


[[12488  2029   226]
 [ 1990 11118  1631]
 [   76  1863 12796]]


              precision    recall  f1-score   support

      budget       0.86      0.85      0.85     14743
         mid       0.74      0.75      0.75     14739
     premium       0.87      0.87      0.87     14735

    accuracy                           0.82     44217
   macro avg       0.82      0.82      0.82     44217
weighted avg       0.82      0.82      0.82     44217



In [2]:
from sklearn.metrics import ConfusionMatrixDisplay  #I'm doing this because ConfusionMatrixDisplay plots a readable heatmap from predictions.

best = clfs["hgb"]  #I'm doing this because histogram gradient boosting was the stronger nonlinear model in the prior cell.
best.fit(X_train, y_train)  #I'm doing this because refitting ensures the displayed matrix matches the latest training run.
pred = best.predict(X_test)  #I'm doing this because predictions on the test set populate the confusion matrix.
fig, ax = plt.subplots(figsize=(5, 4))  #I'm doing this because a compact figure fits the confusion grid neatly.
ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=["budget", "mid", "premium"], ax=ax, colorbar=False)  #I'm doing this because fixed label order aligns rows and columns with tier semantics.
plt.title("HGB confusion matrix")  #I'm doing this because the title identifies which model produced the matrix.
plt.tight_layout()  #I'm doing this because tight_layout prevents axis labels from overlapping the title.
plt.show()  #I'm doing this because show renders the confusion matrix figure inline.


/var/folders/rf/47shdcdn4j522110qmjp300h0000gn/T/ipykernel_25537/3932035544.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  #I'm doing this because show renders the confusion matrix figure inline.


### Binning justification (Slack message draft)

Hey Abishek — I’m binning **listing price** into **3 tiers** using **tertiles** on my cleaned training sample. Thresholds float a bit depending on the exact rows, but the idea is: ~bottom third = **budget**, middle third = **mid**, top third = **premium**. That keeps classes roughly balanced and still maps to “cheap vs normal vs expensive” in plain English.


### Best classifier

**HistGradientBoostingClassifier** edged out logistic regression on accuracy/F1 for me, which matches the intuition that there are nonlinear interactions (miles + age + type isn’t a clean linear boundary).


In [3]:
out = PROJECT_ROOT / "models" / "classification_model.pkl"  #I'm doing this because the classifier artifact should sit beside the regression pickle.
reg = joblib.load(PROJECT_ROOT / "models" / "regression_model.pkl")  #I'm doing this because the regression bundle stores tier cutoffs and category metadata we want to mirror.

bundle = {  #I'm doing this because packaging metadata with the pipeline keeps inference consistent with notebook 02.
    "pipeline": clfs["hgb"],  #I'm doing this because the trained histogram gradient boosting pipeline is the production classifier.
    "feature_cols": FEATURES,  #I'm doing this because the app must supply the same seven columns in order.
    "target": TARGET,  #I'm doing this because downstream checks confirm we predict price tier labels.
    "tier_cutoffs": reg["tier_cutoffs"],  #I'm doing this because identical cutoffs align classification labels with regression bundle documentation.
    "tier_labels": reg["tier_labels"],  #I'm doing this because label order stays synchronized across models.
    "age_base": reg["age_base"],  #I'm doing this because any UI recomputing age uses the same anchor as training.
    "category_levels": reg["category_levels"],  #I'm doing this because valid categorical levels should match the regression artifact.
}  #I'm doing this because closing the dict completes the classification bundle.
joblib.dump(bundle, out, compress=3)  #I'm doing this because compressed joblib keeps artifacts smaller for GitHub and Streamlit Cloud without changing predictions.
print("saved", out.resolve())  #I'm doing this because printing the path confirms where the classifier was written.


saved /Users/jahbrae/Downloads/NEW/models/classification_model.pkl
